In [3]:
import pandas as pd

df = pd.read_csv(r'C:\Users\upadh\Desktop\da_projs\superstore_sales_dashboard\data\Sample - Superstore.csv', 
                 encoding='windows-1252')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (9994, 21)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

First 5 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
import sqlite3
import pandas as pd

# Load into SQLite
conn = sqlite3.connect('superstore.db')
df.to_sql('orders', conn, if_exists='replace', index=False)
print("Database ready!")

Database ready!


In [5]:
query1 = """
SELECT
    ROUND(SUM(Sales), 2) as total_revenue,
    ROUND(SUM(Profit), 2) as total_profit,
    COUNT(DISTINCT "Order ID") as total_orders,
    COUNT(DISTINCT "Customer Name") as total_customers,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) as profit_margin
FROM orders
"""
kpis = pd.read_sql_query(query1, conn)
print("=== KEY KPIs ===")
print(kpis.to_string(index=False))

=== KEY KPIs ===
 total_revenue  total_profit  total_orders  total_customers  profit_margin
    2297200.86     286397.02          5009              793          12.47


In [6]:
query2 = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) as profit_margin,
    COUNT(DISTINCT "Order ID") as total_orders
FROM orders
GROUP BY Category
ORDER BY total_sales DESC
"""
by_category = pd.read_sql_query(query2, conn)
print("=== SALES BY CATEGORY ===")
print(by_category.to_string(index=False))

=== SALES BY CATEGORY ===
       Category  total_sales  total_profit  profit_margin  total_orders
     Technology    836154.03     145454.95          17.40          1544
      Furniture    741999.80      18451.27           2.49          1764
Office Supplies    719047.03     122490.80          17.04          3742


In [7]:
query3 = """
SELECT
    Region,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) as profit_margin,
    COUNT(DISTINCT "Order ID") as total_orders
FROM orders
GROUP BY Region
ORDER BY total_sales DESC
"""
by_region = pd.read_sql_query(query3, conn)
print("=== SALES BY REGION ===")
print(by_region.to_string(index=False))

=== SALES BY REGION ===
 Region  total_sales  total_profit  profit_margin  total_orders
   West    725457.82     108418.45          14.94          1611
   East    678781.24      91522.78          13.48          1401
Central    501239.89      39706.36           7.92          1175
  South    391721.91      46749.43          11.93           822


In [8]:
query4 = """
SELECT
    SUBSTR("Order Date", 1, 7) as month,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    COUNT(DISTINCT "Order ID") as total_orders
FROM orders
GROUP BY month
ORDER BY month
"""
by_month = pd.read_sql_query(query4, conn)
print("=== MONTHLY TREND ===")
print(by_month.to_string(index=False))

=== MONTHLY TREND ===
  month  total_sales  total_profit  total_orders
1/1/201      1481.83       -181.41             4
1/10/20      1247.68       -295.29             4
1/11/20       159.38         28.16             3
1/12/20      1703.13      -1038.71             5
1/13/20      8795.40       1040.60             7
1/14/20      1523.34        219.57             7
1/15/20      2992.15        559.31             9
1/16/20      6632.47       2605.27             4
1/17/20      1390.93       -123.78             5
1/18/20        64.86          6.49             1
1/19/20      2694.05       -254.31             5
1/2/201      4417.57      -1263.05             6
1/20/20      3441.71        740.40             9
1/21/20      3738.95        693.57             6
1/22/20      5130.00       1484.88             9
1/23/20      2272.31        574.64             8
1/24/20       463.15        122.88             4
1/25/20       858.71        -65.85             2
1/26/20      4335.25       1038.33             

In [9]:
query5 = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) as profit_margin
FROM orders
GROUP BY "Sub-Category"
ORDER BY total_sales DESC
LIMIT 10
"""
by_subcategory = pd.read_sql_query(query5, conn)
print("=== TOP SUB-CATEGORIES ===")
print(by_subcategory.to_string(index=False))

=== TOP SUB-CATEGORIES ===
Sub-Category  total_sales  total_profit  profit_margin
      Phones    330007.05      44515.73          13.49
      Chairs    328449.10      26590.17           8.10
     Storage    223843.61      21278.83           9.51
      Tables    206965.53     -17725.48          -8.56
     Binders    203412.73      30221.76          14.86
    Machines    189238.63       3384.76           1.79
 Accessories    167380.32      41936.64          25.05
     Copiers    149528.03      55617.82          37.20
   Bookcases    114880.00      -3472.56          -3.02
  Appliances    107532.16      18138.01          16.87


In [10]:
query6 = """
SELECT
    Segment,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    COUNT(DISTINCT "Customer Name") as customers,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) as profit_margin
FROM orders
GROUP BY Segment
ORDER BY total_sales DESC
"""
by_segment = pd.read_sql_query(query6, conn)
print("=== SALES BY SEGMENT ===")
print(by_segment.to_string(index=False))

=== SALES BY SEGMENT ===
    Segment  total_sales  total_profit  customers  profit_margin
   Consumer   1161401.34     134119.21        409          11.55
  Corporate    706146.37      91979.13        236          13.03
Home Office    429653.15      60298.68        148          14.03


In [11]:
# Save all query results
kpis.to_csv('superstore_kpis.csv', index=False)
by_category.to_csv('by_category.csv', index=False)
by_region.to_csv('by_region.csv', index=False)
by_month.to_csv('by_month.csv', index=False)
by_subcategory.to_csv('by_subcategory.csv', index=False)
by_segment.to_csv('by_segment.csv', index=False)

conn.close()
print("All files saved!")
print("Files ready for Power BI:")
print("  - superstore_kpis.csv")
print("  - by_category.csv")
print("  - by_region.csv")
print("  - by_month.csv")
print("  - by_subcategory.csv")
print("  - by_segment.csv")

All files saved!
Files ready for Power BI:
  - superstore_kpis.csv
  - by_category.csv
  - by_region.csv
  - by_month.csv
  - by_subcategory.csv
  - by_segment.csv


In [12]:
# Fix date format for proper monthly grouping
conn = sqlite3.connect('superstore.db')

query_fixed = """
SELECT
    SUBSTR("Order Date", -4) || '-' || 
    CASE 
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '1' THEN '01'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '2' THEN '02'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '3' THEN '03'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '4' THEN '04'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '5' THEN '05'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '6' THEN '06'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '7' THEN '07'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '8' THEN '08'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '9' THEN '09'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '10' THEN '10'
        WHEN SUBSTR("Order Date", 1, INSTR("Order Date", '/') - 1) = '11' THEN '11'
        ELSE '12'
    END as year_month,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    COUNT(DISTINCT "Order ID") as total_orders
FROM orders
GROUP BY year_month
ORDER BY year_month
"""

by_month_fixed = pd.read_sql_query(query_fixed, conn)
by_month_fixed.to_csv('by_month.csv', index=False)
conn.close()

print("Fixed monthly trend:")
print(by_month_fixed.head(12).to_string(index=False))

Fixed monthly trend:
year_month  total_sales  total_profit  total_orders
   2014-01     14236.90       2450.19            32
   2014-02      4519.89        862.31            28
   2014-03     55691.01        498.73            71
   2014-04     28295.35       3488.84            66
   2014-05     23648.29       2738.71            69
   2014-06     34595.13       4976.52            66
   2014-07     33946.39       -841.48            65
   2014-08     27909.47       5318.11            72
   2014-09     81777.35       8328.10           130
   2014-10     31453.39       3448.26            78
   2014-11     78628.72       9292.13           151
   2014-12     69545.62       8983.57           141
